# Atlético de Madrid 22/23 & 23/24 — os reforços foram substitutos 1:1?

Cruzamos as transferências reais do Atlético (saídas × chegadas, por setor) com o **SPSS**: para cada par *saiu → entrou*, perguntamos **se o sistema considera o reforço parecido com quem saiu** — e por qual lente.

**Método.** Para cada jogador que saiu, calculamos a distância (Euclidiana, z-score **por posição**) a *todos* os 78.283 jogadores de linha e olhamos **em que posição (`rank`) o reforço aparece** na lista de vizinhos do que saiu. Repetimos sob quatro conjuntos de atributos:

- **T** (10) — técnico/estilo de jogo (drible, passe, finalização, primeiro toque…);
- **F** (8) — físico/atlético (aceleração, ritmo, força, resistência, salto…);
- **T+F** (18) — perfil de jogo **+** atlético combinados (a leitura principal);
- **full** (36) — inclui também *mental* e *bola parada* (referência).

`rank` baixo ⇒ substituto direto; alto ⇒ perfil distinto. Comparar **T × F × T+F** separa *semelhança de estilo* de *semelhança física*.

**Faixas de leitura (afrouxadas).** Como os jogadores são próximos, calibramos pelo topo do ranking (de 78.282): 🟢 `≤300` (top ~0,4%) · 🟡 `≤2000` (top ~2,6%) · 🟠 `≤10000` (top ~12,8%) · 🔴 `>10000`. A cor da tabela usa **T+F**.

**Ressalvas.** (1) A normalização é *por posição*: a comparação é mais limpa entre pares da mesma função. (2) **Goleiros ficam fora** do espaço de linha — Grbić → Moldovan (23/24) não é avaliável. (3) *Samu Omorodion* não consta deste FM2023. (4) O dataset é um *snapshot* 22/23 (cada jogador no clube daquele momento; usamos atributos, não clube).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _proj_root():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "data" / "processed" / "outfield_z.csv").exists():
            return c
    raise FileNotFoundError("projeto não encontrado a partir de %s" % Path.cwd())


ROOT = _proj_root()
DATA = ROOT / "data" / "processed"

z = pd.read_csv(DATA / "outfield_z.csv")
meta = (
    pd.read_csv(ROOT / "data" / "merged_players (1).csv", low_memory=False)
    .drop_duplicates("UID")
    .set_index("UID")[["Name", "Club", "Position"]]
)
feat = json.load(open(DATA / "feature_sets.json"))
zU = z.set_index("UID")
present = set(zU.index)
N = len(zU) - 1

# Conjuntos de features analisados: T (técnico/estilo), F (físico), T+F (estilo+físico)
# e full-36 como referência (T+M+F+SP).
allcols = [c for c in z.columns if c not in ("UID", "primary_role")]
SETS = {
    "T": feat["technical"],                 # 10 atributos: drible, passe, finalização...
    "F": feat["physical"],                  # 8 atributos: aceleração, força, resistência...
    "T+F": feat["technical"] + feat["physical"],   # 18 — perfil de jogo + atlético
    "full": allcols,                        # 36 — inclui também mental + bola parada
}


def dist_to_all(uid, cols):
    """Distância Euclidiana (z por posição) de `uid` a todos os jogadores de linha."""
    M = zU[cols].to_numpy(float)
    v = zU.loc[uid, cols].to_numpy(float)
    return pd.Series(np.sqrt(((M - v) ** 2).sum(1)), index=zU.index)


_cache = {}


def pair(out_uid, in_uid, cols):
    """Devolve (distância, rank do reforço entre os vizinhos de quem saiu) sob `cols`."""
    key = (out_uid, len(cols))
    if key not in _cache:
        _cache[key] = dist_to_all(out_uid, cols).drop(out_uid)
    s = _cache[key]
    return float(s[in_uid]), int(s.rank(method="min")[in_uid])


def leitura(rank):
    # Faixas afrouxadas: os jogadores são próximos; aqui o que conta é o topo do ranking.
    if rank <= 300:      # top ~0,4% de 78.282
        return "🟢 substituto direto"
    if rank <= 2000:     # top ~2,6%
        return "🟡 estilo próximo"
    if rank <= 10000:    # top ~12,8%
        return "🟠 cobertura parcial"
    return "🔴 perfil distinto"


print(f"{len(present)} jogadores de linha · base do ranking: N={N}")
print("Conjuntos:", {k: len(v) for k, v in SETS.items()})
print("Fora do espaço de linha (não avaliáveis): Ivo Grbić e H. Moldovan (goleiros); Samu Omorodion (ausente).")

78283 jogadores de linha · base do ranking: N=78282
Conjuntos: {'T': 10, 'F': 8, 'T+F': 18, 'full': 36}
Fora do espaço de linha (não avaliáveis): Ivo Grbić e H. Moldovan (goleiros); Samu Omorodion (ausente).


In [2]:
# (temporada, setor, saiu, UID_saiu, entrou, UID_entrou) — pares saída→chegada por setor.
# Setores sem chegada direta (zaga 22/23, reforços de zaga 23/24, Camello→Omorodion) não entram aqui.
PARES = [
    ("22/23", "Lateral-direito",  "Vrsaljko",      24004437, "Nahuel Molina", 14129337),
    ("22/23", "Lateral-direito",  "S. Arias",      76015950, "Nahuel Molina", 14129337),
    ("22/23", "Lateral-direito",  "D. Wass",         936891, "Nahuel Molina", 14129337),
    ("22/23", "Lateral-direito",  "Vrsaljko",      24004437, "Matt Doherty",  52032335),
    ("22/23", "Lateral-esquerdo", "Renan Lodi",    19259027, "Reguilón",      67211900),
    ("22/23", "Lateral-esquerdo", "Manu Sánchez",  67284340, "Reguilón",      67211900),
    ("22/23", "Volante",          "H. Herrera",     5664830, "Witsel",         8169215),
    ("22/23", "Centroavante",     "L. Suárez",     78000335, "Depay",         37024470),
    ("22/23", "Ataque (jan)",     "João Félix",    83169822, "Depay",         37024470),
    ("22/23", "Ataque (jan)",     "Matheus Cunha", 19334410, "Depay",         37024470),
    ("23/24", "Lateral-direito",  "Matt Doherty",  52032335, "Azpilicueta",   67033300),
    ("23/24", "Meio-campo",       "Kondogbia",     85033422, "Vermeeren",   2000127465),
    ("23/24", "Lateral-esquerdo", "Renan Lodi",    19259027, "Javi Galán",    67243881),
    ("23/24", "Lateral-esquerdo", "Manu Sánchez",  67284340, "Javi Galán",    67243881),
    ("23/24", "Ataque",           "Matheus Cunha", 19334410, "Griezmann",     67086656),
    ("23/24", "Ataque",           "João Félix",    83169822, "Griezmann",     67086656),
    ("23/24", "Ataque",           "Carrasco",      18027198, "Griezmann",     67086656),
]

# rank do reforço entre os vizinhos de quem saiu, sob cada conjunto de features.
# A leitura (verde/amarelo/...) usa T+F = perfil de jogo + atlético.
linhas = []
for temp, setor, ol, ou, il, iu in PARES:
    rk = {t: pair(ou, iu, SETS[t])[1] for t in ("T", "F", "T+F", "full")}
    func = f"{zU.loc[ou, 'primary_role']}→{zU.loc[iu, 'primary_role']}"
    linhas.append([temp, setor, ol, il, func,
                   rk["T"], rk["F"], rk["T+F"], rk["full"], leitura(rk["T+F"])])

transfers = pd.DataFrame(linhas, columns=[
    "temp", "setor", "saiu", "entrou", "func",
    "rank_T", "rank_F", "rank_T+F", "rank_full", "leitura (T+F)"])
transfers

,temp,setor,saiu,entrou,func,rank_T,rank_F,rank_T+F,rank_full,leitura (T+F)
0,22/23,Lateral-direito,Vrsaljko,Nahuel Molina,DEF→DEF,34299,25004,27491,15647,🔴 perfil distinto
1,22/23,Lateral-direito,S. Arias,Nahuel Molina,DEF→DEF,3400,102,285,185,🟢 substituto direto
2,22/23,Lateral-direito,D. Wass,Nahuel Molina,DEF→DEF,17,29872,585,40,🟡 estilo próximo
3,22/23,Lateral-direito,Vrsaljko,Matt Doherty,DEF→DEF,27042,10248,16704,5335,🔴 perfil distinto
4,22/23,Lateral-esquerdo,Renan Lodi,Reguilón,DEF→DEF,152,50,14,36,🟢 substituto direto
5,22/23,Lateral-esquerdo,Manu Sánchez,Reguilón,DEF→DEF,2388,7304,1997,293,🟡 estilo próximo
6,22/23,Volante,H. Herrera,Witsel,MID→DEF,15948,994,5077,2551,🟠 cobertura parcial
7,22/23,Centroavante,L. Suárez,Depay,FW→AM,2223,74107,15773,1623,🔴 perfil distinto
8,22/23,Ataque (jan),João Félix,Depay,AM→AM,138,40154,1765,539,🟡 estilo próximo
9,22/23,Ataque (jan),Matheus Cunha,Depay,AM→AM,131,1975,94,346,🟢 substituto direto


In [3]:
# O que o PRÓPRIO sistema sugeriria como substituto (top-5 vizinhos), sob cada conjunto.
# Comparar T x F x T+F mostra se a semelhança é de ESTILO, de FÍSICO, ou de ambos.
def top5(uid, cols):
    t = dist_to_all(uid, cols).drop(uid).sort_values().head(5)
    return " | ".join(f"{meta.loc[i, 'Name']} ({zU.loc[i, 'primary_role']}, {t[i]:.2f})" for i in t.index)


for lab, u in [
    ("L. Suárez", 78000335), ("João Félix", 83169822), ("Matheus Cunha", 19334410),
    ("Renan Lodi", 19259027), ("Kondogbia", 85033422), ("Vrsaljko", 24004437),
]:
    print(f"### {lab}")
    for t in ("T", "F", "T+F"):
        print(f"  [{t:>3}] {top5(u, SETS[t])}")
    print()

### L. Suárez
  [  T] Kevin De Bruyne (MID, 1.99) | Gonzalo Higuaín (FW, 2.27) | Fabián (DM, 2.34) | Rodrigo De Paul (MID, 2.44) | Hakan Çalhanoğlu (MID, 2.46)
  [  F] Lee Ho-Bin (MID, 0.66) | Vinni Triboulet (AM, 0.82) | Julen Bernaola (DEF, 0.87) | Massimo D'Angelo (MID, 0.91) | Kirill Nababkin (DEF, 0.92)
  [T+F] Dani Parejo (DM, 3.20) | Gonzalo Higuaín (FW, 3.28) | Sérgio Oliveira (DM, 3.41) | Claudio Bieler (FW, 3.48) | Aaron Mooy (DM, 3.61)

### João Félix
  [  T] Marko Arnautović (FW, 1.38) | Giorgian De Arrascaeta (MID, 1.61) | Sergio Canales (MID, 1.61) | Isco (MID, 1.73) | Stevan Jovetić (AM, 1.78)
  [  F] Tomás Andrade (AM, 0.60) | Ignacio Colombini (FW, 0.63) | Exequiel Palacios (DM, 0.64) | Albert Rusnák (AM, 0.67) | Leonardo Villán (DEF, 0.73)
  [T+F] Giovani Lo Celso (DM, 2.67) | Rúben Neves (DM, 2.74) | Rachid Ghezzal (MID, 2.75) | Sergio Canales (MID, 2.75) | Giorgian De Arrascaeta (MID, 2.77)

### Matheus Cunha
  [  T] Aleksandar Katai (AM, 1.42) | Leroy Sané (MID, 1.

## Leitura dos resultados (T × F × T+F)

Separar técnica (estilo) de físico muda — e enriquece — o veredito. Com as faixas afrouxadas e a lente **T+F**, os substitutos diretos finalmente aparecem em **verde**.

**Laterais.** **Reguilón ≈ Lodi** é um clone robusto: rank 152 (T), **50 (F)** e **14 (T+F)** — verde. A "troca direta" de 22/23 se confirma sob qualquer lente. Em 23/24 o substituto piora: Javi Galán fica em 🟡/🟠 (T+F ~1.150 p/ Lodi, ~5.500 p/ Manu) — a saída "em duas etapas" foi de *clone* (Reguilón) para *cobertura* (Galán). **Nahuel Molina** mostra o valor da decomposição: é **gêmeo técnico do Wass** (T rank **17**) mas **gêmeo atlético do Arias** (F rank **102**) — dois laterais distintos cujos perfis, somados, ele cobre; já o marcador **Vrsaljko** está longe dele em todas as lentes (🔴).

**Volante / meio.** **Vermeeren** (17 anos) por Kondogbia é o "não-match" mais nítido: 🔴 em tudo (T 23.297, F 57.788, T+F 39.037) — "veterano por jovem promessa" não é reposição. Witsel por Herrera é próximo no **físico** (F rank 994) mas longe na técnica — coerente com o volante que recuou para a zaga.

**Ataque — o ponto que a decomposição revela.** Sob o full-36 os reforços pareciam distantes, mas isso vinha do *ruído mental/bola-parada*. Em **T+F** três pares sobem para **verde**:

- **Matheus Cunha → Depay**: T 131, **T+F 94** 🟢 — estilo e físico muito parecidos;
- **Matheus Cunha → Griezmann**: **T+F 158** 🟢 — também substituto direto de perfil;
- **João Félix → Griezmann**: T 45, **T+F 25** 🟢 — gêmeos de estilo (físico diferente, F 3.360).

E ficam logo atrás **João Félix → Depay** (🟡, T+F 1.765) e **Carrasco → Griezmann** (🟠) — parentescos mais fracos.

Ou seja: as chegadas do ataque **eram** compatíveis em perfil de jogo+físico — só que **espalhadas**. O clube repôs o *perfil coletivo* do ataque, não um substituto 1:1 por craque que saiu (Depay e Griezmann cobrem, juntos, os estilos de Cunha e Félix). A exceção que continua "venda sem reposição" é **L. Suárez → Depay**: tecnicamente moderado (T 2.223) e fisicamente oposto (**F 74.107** — Suárez é um 9 de área, Depay não), confirmando que ali não houve substituto de perfil.

**Síntese.** (a) O buraco da lateral-esquerda foi de **troca-direta** (Reguilón, verde em T+F) a **cobertura-parcial** (Galán). (b) O reinvestimento no ataque é **difuso, mas estilisticamente coerente**: sob T+F os reforços são parentes próximos dos que saíram — só que distribuídos, não 1:1. (c) "Veterano por jovem" (Vermeeren) e o 9 de área (Suárez) são os únicos casos sem qualquer parentesco de perfil.